# CSE 153/253 Assignment 2 Workbook

## Project Goal

We chose **Task 1: symbolic unconditioned generation** and **Task 2: symbolic conditioned generation**.

The shared idea is to treat music as a sequence of discrete symbolic tokens, similar to a small language model for folk melodies. The dataset is the Nottingham folk tune collection in ABC notation. From each tune we extract melody notes, rests, bar boundaries, durations, and chord symbols. Then we train models from scratch to predict the next token.

- **Task 1 output:** `symbolic_unconditioned.mid`, generated from only `<START>`.
- **Task 2 output:** `symbolic_conditioned.mid`, generated from an explicit chord progression.

The main model is a GRU next-token predictor. The main baseline is a trigram model, which is useful because it captures local note transitions but has no learned hidden state.

In [ ]:
from pathlib import Path
import json
import sys

from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


def load_json(relative_path):
    path = ROOT / relative_path
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_jsonl(relative_path, limit=None):
    rows = []
    path = ROOT / relative_path
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
            if limit and len(rows) >= limit:
                break
    return rows


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(item) for item in row) + " |")
    return "\n".join(lines)


def fmt(value, digits=3):
    if value is None:
        return "n/a"
    if isinstance(value, float):
        return f"{value:.{digits}f}"
    return value

print(f"Project root: {ROOT}")

## Shared Dataset, Preprocessing, And Exploratory Analysis

**Context.** Nottingham is a collection of folk tunes distributed in ABC notation. It is a good fit because it contains monophonic melody information and chord labels in a compact symbolic format. This directly supports both selected tasks: melody-only generation and chord-conditioned melody generation.

**Preprocessing.** The pipeline downloads the ABC files, parses them with `music21`, selects the first melody part, quantizes durations to sixteenth-note units, and writes JSONL train/validation splits. The unconditioned stream contains `<START>`, `NOTE_pitch_duration`, `REST_duration`, `<BAR>`, and `<END>`. The conditioned stream inserts `CHORD_*` tokens before measures so the GRU state carries harmonic context while predicting melody tokens.

The code for this section is in `src/preprocessing/tokenize_nottingham.py` and `src/analysis/dataset_summary.py`.

In [ ]:
dataset = load_json("outputs/metrics/dataset_summary.json")
rows = []
for split_name in ["train_unconditioned", "val_unconditioned", "train_conditioned", "val_conditioned"]:
    split = dataset["splits"][split_name]
    rows.append([
        split_name,
        split["num_sequences"],
        split["num_tokens"],
        split["vocab_size"],
        fmt(split["sequence_length"]["mean"], 1),
        split["unique_pitches"],
        split["pitch_range"],
        split["unique_chords"],
    ])

display(Markdown(md_table(
    ["split", "tunes", "tokens", "vocab", "avg length", "unique pitches", "pitch range", "unique chords"],
    rows,
)))

In [ ]:
example = load_jsonl("data/processed/nottingham/train_conditioned.jsonl", limit=1)[0]
example_tokens = example["tokens"][:32]

rows = [[idx, token] for idx, token in enumerate(example_tokens)]
display(Markdown("### Concrete Tokenization Example"))
display(Markdown(f"Source tune: `{example['source']}`"))
display(Markdown(md_table(["position", "token"], rows)))
display(Markdown(
    "This example shows the actual representation used by both models: chord tokens mark harmonic context, "
    "note tokens encode MIDI pitch and duration, and `<BAR>` marks phrase/measure boundaries."
))

In [ ]:
figure_titles = {
    "sequence_lengths": "Tokenized Tune Lengths",
    "pitch_distribution": "Melody Pitch Distribution",
    "duration_distribution": "Duration Token Distribution",
    "top_note_tokens": "Most Common Note Tokens",
    "top_chords": "Most Common Conditioning Chords",
}

for key, title in figure_titles.items():
    figure = dataset["figures"].get(key)
    if figure:
        display(Markdown(f"### {title}"))
        display(Image(filename=str(ROOT / figure)))

## Modeling Approach

Both tasks are formulated as next-token prediction. For a token sequence `x_1, ..., x_T`, the GRU is trained to minimize cross-entropy for `p(x_t | x_<t)`. During generation, we sample one token at a time from the model distribution using temperature and top-k sampling.

**Why a GRU?** A GRU is more expressive than an n-gram model because it keeps a continuous hidden state over long contexts, but it is still small enough to train from scratch on Nottingham. A transformer would be a reasonable extension, but it adds implementation and compute cost that is not needed for this assignment.

**Baseline.** The trigram model is intentionally simple. It estimates the next-token distribution from the previous two tokens. This is a fair trivial baseline because it captures local musical syntax but cannot learn longer phrase structure or flexible chord conditioning.

Relevant code:

- GRU architecture: `src/models/gru_model.py`
- Training loop: `src/training/train_gru.py`
- N-gram baseline: `src/models/baselines.py`
- Sampling and MIDI writing: `src/generation/sample_gru.py`, `src/generation/midi_writer.py`

In [ ]:
training = load_json("outputs/metrics/training_summary.json")
rows = []
for task in ["unconditioned", "conditioned"]:
    gru = training["gru_models"][task]
    baseline = training["baselines"][f"ngram_{task}"]
    rows.append([
        task,
        baseline["n"],
        fmt(baseline["validation_perplexity"]),
        gru["epochs"],
        gru["best_epoch"],
        fmt(gru["best_val_perplexity"]),
        fmt(baseline["validation_perplexity"] / gru["best_val_perplexity"], 2) + "x",
        fmt(gru["final_generalization_gap"]),
    ])

display(Markdown(md_table(
    ["task", "baseline n", "n-gram val ppl", "GRU epochs", "best epoch", "GRU best val ppl", "baseline/GRU ppl", "final val-train gap"],
    rows,
)))

In [ ]:
for figure, title in [
    ("figures/training_curves.png", "GRU Loss Curves"),
    ("figures/perplexity_comparison.png", "Baseline Versus GRU Perplexity"),
]:
    display(Markdown(f"### {title}"))
    display(Image(filename=str(ROOT / figure)))

**Modeling interpretation.** The validation losses decrease quickly and then flatten, which is what we want for a compact dataset: the GRUs learn useful sequence structure without requiring a large training run. The perplexity comparison is the clearest quantitative result. The unconditioned GRU is about 3.4x lower perplexity than its trigram baseline, and the conditioned GRU is about 4.4x lower than its trigram baseline. That gap supports the modeling choice: longer learned context helps more than short local counts alone.

## Evaluation Protocol

The objective function is cross-entropy next-token prediction, reported as validation perplexity. Lower perplexity means the model assigns higher probability to held-out Nottingham token sequences. Musical quality is broader than perplexity, so we also inspect generated samples with surface-level metrics:

- **Pitch range and pitch diversity:** checks whether the sample stays in a plausible register without collapsing to one or two notes.
- **Duration diversity:** checks rhythmic variety.
- **Repetition rate:** flags immediate loops.
- **Large leap rate:** checks whether melodic motion is mostly stepwise/singable or jumpy.
- **Chord-tone rate for Task 2:** checks whether generated notes align with the conditioning chords.

These metrics are not a substitute for listening. Passing tones, suspensions, repetition, and leaps can all be musically valid in context. The metrics are best understood as sanity checks that support the final listening demo.

## Task 1: Symbolic Unconditioned Generation

**Task definition.** The model learns a melody distribution `p(x)` from Nottingham melody tokens. At generation time, it receives only `<START>` and samples a complete melody until `<END>` or a maximum length.

**Inputs and outputs.** Input is the previous melody-token context. Output is the next melody token. The final audio artifact is rendered as `symbolic_unconditioned.mid`.

**Evaluation goals.** A good unconditioned output should stay in a plausible folk register, avoid immediate loops, contain varied pitches/durations, and sound like a coherent melody. Validation perplexity measures predictive quality, while generated-token metrics measure musical surface properties. These metrics are imperfect, so listening remains important.

In [ ]:
unconditioned = load_json("outputs/metrics/unconditioned_generation.json")
ngram_unconditioned = training["baselines"]["ngram_unconditioned"]["sample_metrics"]
gen = unconditioned["generated"]
ref = unconditioned["reference"]
rows = [
    ["generated GRU", gen["num_note_tokens"], gen["unique_pitches"], gen["pitch_range"], gen["unique_durations"], fmt(gen["pitch_class_entropy"]), fmt(gen["repetition_rate"]), fmt(gen["large_leap_rate"])],
    ["generated n-gram", ngram_unconditioned["num_note_tokens"], ngram_unconditioned["unique_pitches"], ngram_unconditioned["pitch_range"], ngram_unconditioned["unique_durations"], fmt(ngram_unconditioned["pitch_class_entropy"]), fmt(ngram_unconditioned["repetition_rate"]), fmt(ngram_unconditioned["large_leap_rate"])],
    ["validation reference", fmt(ref["notes_per_sequence"]["mean"], 1), ref["unique_pitches"], ref["pitch_range"], ref["unique_durations"], fmt(ref["pitch_class_entropy"]), "n/a", "n/a"],
]

display(Markdown(md_table(
    ["source", "notes", "unique pitches", "pitch range", "unique durations", "pitch-class entropy", "repeat rate", "large leap rate"],
    rows,
)))

display(Markdown("**Most common generated GRU notes:** " + ", ".join(f"`{token}` ({count})" for token, count in gen["top_notes"][:8])))

**Task 1 interpretation.** The GRU validation perplexity is much lower than the trigram baseline, which shows that the recurrent hidden state is learning useful sequential structure beyond short local transitions. The generated sample uses a realistic pitch range and low repetition rate. Its duration diversity is lower than the validation set, which is a limitation: the model favors common short note lengths.

**Listening notes to discuss in the presentation.** For `symbolic_unconditioned.mid`, listen for whether phrases feel folk-like, whether bar boundaries produce sensible grouping, and whether the repeated common pitches sound like stylistic consistency or like model overconfidence. The metric table predicts a mostly stepwise melody with moderate pitch variety and little immediate token repetition.

## Task 2: Symbolic Chord-Conditioned Generation

**Task definition.** The model learns `p(melody | chords)` by training on token streams where each measure begins with a chord token. At generation time, we provide a chord progression and sample melody tokens after each chord.

**Inputs and outputs.** Input is the previous token context plus the active chord token. Output is the next melody token. The final artifact is `symbolic_conditioned.mid`.

**Evaluation goals.** A good conditioned melody should still have melodic variety, but it should also fit the chord progression. The extra metric here is chord-tone rate: among generated notes with an active chord, the fraction whose pitch class belongs to a simple chord-tone set inferred from the chord symbol.

In [ ]:
conditioned = load_json("outputs/metrics/conditioned_generation.json")
ngram_conditioned = training["baselines"]["ngram_conditioned"]["sample_metrics"]
gen = conditioned["generated"]
ref = conditioned["reference"]
rows = [
    ["generated GRU", gen["num_note_tokens"], gen["num_chord_tokens"], gen["unique_pitches"], gen["pitch_range"], gen["unique_durations"], fmt(gen["pitch_class_entropy"]), fmt(gen["chord_tone_rate"]), fmt(gen["large_leap_rate"])],
    ["generated n-gram", ngram_conditioned["num_note_tokens"], ngram_conditioned["num_chord_tokens"], ngram_conditioned["unique_pitches"], ngram_conditioned["pitch_range"], ngram_conditioned["unique_durations"], fmt(ngram_conditioned["pitch_class_entropy"]), fmt(ngram_conditioned["chord_tone_rate"]), fmt(ngram_conditioned["large_leap_rate"])],
    ["validation reference", fmt(ref["notes_per_sequence"]["mean"], 1), ref["num_chord_tokens"], ref["unique_pitches"], ref["pitch_range"], ref["unique_durations"], fmt(ref["pitch_class_entropy"]), "n/a", "n/a"],
]

display(Markdown(md_table(
    ["source", "notes", "chord tokens", "unique pitches", "pitch range", "unique durations", "pitch-class entropy", "chord-tone rate", "large leap rate"],
    rows,
)))

chord_rows = []
for chord, values in gen["chord_tone_by_chord"].items():
    chord_rows.append([chord, values["notes"], fmt(values["chord_tone_rate"])])

display(Markdown("### Chord Fit By Conditioning Chord"))
display(Markdown(md_table(["chord", "checked notes", "chord-tone rate"], chord_rows)))

**Task 2 interpretation.** The conditioned GRU also beats the conditioned trigram baseline in validation perplexity. The generated sample includes the supplied chord tokens and achieves a chord-tone rate above 0.7, meaning most notes land on simple chord tones. This metric is not the same as musical quality: non-chord tones can sound good as passing tones, neighbor tones, or suspensions. Still, it is a useful sanity check that the conditioning signal is influencing the melody.

**Listening notes to discuss in the presentation.** For `symbolic_conditioned.mid`, listen at chord changes. The main question is whether the melody feels anchored to G/D/C/Em while still moving naturally. The chord-tone table predicts the strongest fit over G and D, with C and Em leaving more room for passing tones or occasional mismatch.

## Generated Files

The final top-level files are the required assignment deliverables for the two selected tasks. Baseline MIDI files are kept under `outputs/midi/` for comparison and discussion, but the submitted task files should be the GRU outputs.

In [ ]:
from mido import MidiFile

rows = []
for relative_path in [
    "symbolic_unconditioned.mid",
    "symbolic_conditioned.mid",
    "outputs/midi/ngram_unconditioned.mid",
    "outputs/midi/ngram_conditioned.mid",
]:
    path = ROOT / relative_path
    midi = MidiFile(path)
    notes = sum(1 for track in midi.tracks for msg in track if msg.type == "note_on" and msg.velocity > 0)
    rows.append([relative_path, path.stat().st_size, fmt(midi.length, 1), notes])

display(Markdown(md_table(["file", "bytes", "seconds", "note_on events"], rows)))

## Related Work Discussion

This project follows the common symbolic-music language-modeling setup: encode music as discrete events, train a sequence model, then sample new event sequences.

- The [ABC version of the Nottingham Music Database](https://abc.sourceforge.net/NMD/) describes Nottingham as a collection of more than 1000 folk tunes in a text notation format. That matches our choice of ABC parsing and symbolic tokenization.
- Eck and Schmidhuber's [LSTM blues improvisation work](https://research.google/pubs/finding-temporal-structure-in-music-blues-improvisation-with-lstm-recurrent-networks/) is an early example of using recurrent networks to model musical temporal structure.
- Boulanger-Lewandowski, Bengio, and Vincent's [polyphonic music sequence model](https://arxiv.org/abs/1206.6392) frames symbolic music as a probabilistic sequence modeling problem, close in spirit to our next-token setup.
- Huang et al.'s [Music Transformer](https://arxiv.org/abs/1809.04281) shows how attention-based models can improve long-range musical structure and can also support conditioned generation, such as accompaniment conditioned on melody.

Compared with those larger or more complex systems, our GRU is intentionally modest: it trains quickly from scratch, is easy to explain in a 20-minute presentation, and still clearly improves over n-gram baselines. The main limitation is long-range form. Our generated files are phrase-length folk-style samples, not full compositions with repeated sections, development, or global structure.

## What To Say In The Presentation

A concise presentation path is:

1. We chose two symbolic tasks: unconditioned melody generation and chord-conditioned melody generation.
2. We used Nottingham because it has ABC melodies plus chord symbols, which directly supports both tasks.
3. We tokenized notes, rests, durations, bars, and chords, then trained GRU next-token models from scratch.
4. We compared against trigram baselines. The GRUs achieved much lower validation perplexity.
5. We evaluated generated samples with diversity, repetition, pitch range, leap rate, and chord-tone metrics.
6. We should play the two final MIDI files at the end: `symbolic_unconditioned.mid` and `symbolic_conditioned.mid`.

Before recording the final video, listen to both files once and add one spoken sentence about what sounds strongest and one sentence about the most obvious limitation. That small subjective reflection helps satisfy the evaluation part of the rubric.